In [1]:
#load all the necessary libraries
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from rdkit import Chem
from rdkit.Chem import Descriptors


In [43]:
#Datasets
#load the datasets 
df_delaney = pd.read_csv(Path("data/delaney.csv"))
df_mic = pd.read_csv(Path("data/ecoli_mics_sample.csv"))

#inspect the datasets
print("head delaney:",df_delaney.head())
print("head mic:", df_mic.head())
print("delaney statistics:",df_delaney.describe())
print("mic statistics:",df_mic.describe())

#check for missing values and duplicates: 
print("duplicates and missing values in delaney:",df_delaney.duplicated().sum(), df_delaney.isnull().sum()) #no duplicates or missing values in delaney dataset
print("duplicates and missing values in mic:", df_mic.duplicated().sum(), df_mic.isnull().sum()) #two missing values in mic dataset, no duplicates

#check if units for mic are consistent:
print("units in mic dataset:", df_mic['unit'].unique()) #units are consistent. 

#drop the missing values in mic dataset
df_mic.dropna(inplace=True)
df_mic.info()

head delaney:                       Compound ID  logS               smiles
0       1,1,1,2-Tetrachloroethane -2.18       ClCC(Cl)(Cl)Cl
1           1,1,1-Trichloroethane -2.00         CC(Cl)(Cl)Cl
2       1,1,2,2-Tetrachloroethane -1.74       ClC(Cl)C(Cl)Cl
3           1,1,2-Trichloroethane -1.48           ClCC(Cl)Cl
4  1,1,2-Trichlorotrifluoroethane -3.04  FC(F)(Cl)C(F)(Cl)Cl
head mic:        chembl_id                                             smiles    mic  \
0  CHEMBL1689756  CN(C)[C@@H]1C(O)=C(C(N)=O)C(=O)[C@@]2(O)C(O)=C...   0.25   
1  CHEMBL1222951  CO/N=C1/CCN(c2c(F)cc3c(=O)c(C(=O)O)cn(C4CC4)c3...   8.00   
2  CHEMBL2403434          COc1cccc(CC(=O)Nc2nc(-c3ccc(Br)cc3)cs2)c1  12.50   
3      CHEMBL148  C[C@@H](O)[C@H]1C(=O)N2C(C(=O)O)=C(SCCNC=N)C[C...   0.19   
4       CHEMBL22                    COc1cc(Cc2cnc(N)nc2N)cc(OC)c1OC   0.25   

      unit  
0  ug.mL-1  
1  ug.mL-1  
2  ug.mL-1  
3  ug.mL-1  
4  ug.mL-1  
delaney statistics:               logS
count  1144.000000
mean 

In [44]:
# function for adding images from RDKit to the datasets:
def add_molecule_images(df): 
    mol_list = df['smiles'].apply(lambda s: Chem.MolFromSmiles(s))
    df['image']= mol_list.apply(lambda m: Chem.Draw.MolToImage(m, size=(200, 200)))
    return df

In [45]:
# function for adding customed fingerprint column to the datasets:
from rdkit.Chem import rdFingerprintGenerator
def add_custom_fingerprints(df): 
    mol_list = df['smiles'].apply(Chem.MolFromSmiles)
    motifs = [
        "[a]",
        "[OX2H]", 
        "[NX3]",
        "C=C",
        "C#C",
        "[CX3]=O[OX2]",
        "C(=O)C", 
        "[F,Cl,Br,I]"
    ] # add aromatic, hydroxyl, amine, double bond, ester, ketone, halogen
    df['custom_fingerprint'] = None
    for idx, mol in enumerate(mol_list):
        fingerprint =[] 
        for smarts in motifs: 
            pattern = Chem.MolFromSmarts(smarts)
            if mol.HasSubstructMatch(pattern):
                fingerprint.append(1)
            else:
                fingerprint.append(0)
        df.at[idx, 'custom_fingerprint'] = fingerprint

    return df

In [46]:
#apply the functions to the dataset 
df_delaney = add_custom_fingerprints(df_delaney)
#print(df_delaney_images.head())

#df_delaney_fingerprints = add_fingerprits(df_delaney)

df_delaney.head()

,Compound ID,logS,smiles,custom_fingerprint
0,"1,1,1,2-Tetrachloroethane",-2.18,ClCC(Cl)(Cl)Cl,"[0, 0, 0, 0, 0, 0, 0, 1]"
1,"1,1,1-Trichloroethane",-2.00,CC(Cl)(Cl)Cl,"[0, 0, 0, 0, 0, 0, 0, 1]"
2,"1,1,2,2-Tetrachloroethane",-1.74,ClC(Cl)C(Cl)Cl,"[0, 0, 0, 0, 0, 0, 0, 1]"
3,"1,1,2-Trichloroethane",-1.48,ClCC(Cl)Cl,"[0, 0, 0, 0, 0, 0, 0, 1]"
4,"1,1,2-Trichlorotrifluoroethane",-3.04,FC(F)(Cl)C(F)(Cl)Cl,"[0, 0, 0, 0, 0, 0, 0, 1]"


In [47]:
# function for adding Morgan fingerprint column to the datasets:

def add_morgan_fingerprints(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=2048)
    fp = mfpgen.GetFingerprint(mol)
    return fp


In [48]:
#apply the function to the datasets
df_delaney["morgan_fingerprint"] = df_delaney["smiles"].apply(add_morgan_fingerprints)

df_delaney.head()

,Compound ID,logS,smiles,custom_fingerprint,morgan_fingerprint
0,"1,1,1,2-Tetrachloroethane",-2.18,ClCC(Cl)(Cl)Cl,"[0, 0, 0, 0, 0, 0, 0, 1]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,"1,1,1-Trichloroethane",-2.00,CC(Cl)(Cl)Cl,"[0, 0, 0, 0, 0, 0, 0, 1]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,"1,1,2,2-Tetrachloroethane",-1.74,ClC(Cl)C(Cl)Cl,"[0, 0, 0, 0, 0, 0, 0, 1]","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,"1,1,2-Trichloroethane",-1.48,ClCC(Cl)Cl,"[0, 0, 0, 0, 0, 0, 0, 1]","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,"1,1,2-Trichlorotrifluoroethane",-3.04,FC(F)(Cl)C(F)(Cl)Cl,"[0, 0, 0, 0, 0, 0, 0, 1]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [49]:
# calculate RDKit descriptors and store in a new dataframe 

def calculate_descriptors (df): 
    mol_list = df['smiles'].apply(Chem.MolFromSmiles)
    descriptors = []
    for mol in mol_list:
        descriptors.append (Descriptors.CalcMolDescriptors(mol))
    df_descriptors = pd.DataFrame(descriptors)
    return df_descriptors

In [26]:
#apply to the dataframe 

df_delaney_descriptors = calculate_descriptors(df_delaney)
df_delaney_descriptors.head()

,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,5.116512,5.116512,0.039352,-1.276235,0.487138,12.00,167.850,165.834,165.891061,38,...,0,0,0,0,0,0,0,0,0,0
1,5.060957,5.060957,1.083333,-1.083333,0.445171,12.00,133.405,130.381,131.930033,32,...,0,0,0,0,0,0,0,0,0,0
2,5.114198,5.114198,0.672840,-0.672840,0.527312,11.00,167.850,165.834,165.891061,38,...,0,0,0,0,0,0,0,0,0,0
3,5.095679,5.095679,0.308642,-0.405864,0.480258,9.60,133.405,130.381,131.930033,32,...,0,0,0,0,0,0,0,0,0,0
4,11.544753,11.544753,3.685957,-4.226080,0.553756,14.25,187.375,187.375,185.901768,50,...,0,0,0,0,0,0,0,0,0,0


In [50]:
#compare custom fingerprint, Morgan fingerprint and physical descriptors in a Random Forest model: 

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

# function to train and evaluate Random Forest: 
def apply_random_forest (df, feature_column, target_column, feature_is_column=True):
    if feature_is_column:
        X = np.vstack(df[feature_column])  # for descriptors, you can simply use X = descr_df            
    else:
        X = feature_column # for the case of descriptor dataframe 
    y = df[target_column].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

    rf = RandomForestRegressor(n_estimators=200, random_state=42)
    rf.fit(X_train, y_train)

    pred_train = rf.predict(X_train)
    pred_test = rf.predict(X_test)

    print("Train R2 on", feature_column, ":", r2_score(y_train, pred_train))
    print("Train R2 on", feature_column, ":", r2_score(y_train, pred_train))
    print("Test  R2 on", feature_column, ":", r2_score(y_test, pred_test))
    print("Test RMSE on", feature_column, ":", mean_squared_error(y_test, pred_test) ** 0.5)
    print("Test MAE on", feature_column, ":", mean_absolute_error(y_test, pred_test))

In [39]:
#apply different models to delaney dataset: 


apply_random_forest(df_delaney, 'custom_fingerprint', 'logS', feature_is_column=True)
apply_random_forest(df_delaney, 'morgan_fingerprint', 'logS', feature_is_column=True) 
#apply_random_forest(df_delaney, df_delaney_descriptors, 'logS', feature_is_column=False)

## Model with physical desctriptors didn't work as intended > copied function again. 

Train R2 on custom_fingerprint : 0.32990307814220965
Train R2 on custom_fingerprint : 0.32990307814220965
Test  R2 on custom_fingerprint : 0.2782580275246289
Test RMSE on custom_fingerprint : 1.77281023165941
Test MAE on custom_fingerprint : 1.3540875806692474
Train R2 on morgan_fingerprint : 0.9430029963649174
Train R2 on morgan_fingerprint : 0.9430029963649174
Test  R2 on morgan_fingerprint : 0.6995607796691561
Test RMSE on morgan_fingerprint : 1.1437986552327435
Test MAE on morgan_fingerprint : 0.8558554351094074


In [38]:
X = df_delaney_descriptors     # for the case of descriptor dataframe 
y = df_delaney['logS'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

pred_train = rf.predict(X_train)
pred_test = rf.predict(X_test)

print("Train R2 on physical descriptors", r2_score(y_train, pred_train))
print("Train R2 on physical descriptors", r2_score(y_train, pred_train))
print("Test  R2 on physical descriptors", r2_score(y_test, pred_test))
print("Test RMSE on physical descriptors", mean_squared_error(y_test, pred_test) ** 0.5)
print("Test MAE on physical descriptors", mean_absolute_error(y_test, pred_test))

Train R2 on physical descriptors 0.9865213445364112
Train R2 on physical descriptors 0.9865213445364112
Test  R2 on physical descriptors 0.9099577890472579
Test RMSE on physical descriptors 0.6261730091748434
Test MAE on physical descriptors 0.45846544739031003


In [ ]:
# interpretation of the delaney results: 
# summary: 
# custom fingerprint: 
##Test  R2 on custom_fingerprint : 0.2782580275246289
##Test RMSE on custom_fingerprint : 1.77281023165941
##Test MAE on custom_fingerprint : 1.3540875806692474
# Morgan fingerprint: 
##Test  R2 on morgan_fingerprint : 0.6995607796691561
##Test RMSE on morgan_fingerprint : 1.1437986552327435
##Test MAE on morgan_fingerprint : 0.8558554351094074
# Physical descriptors: 
##Test  R2 on physical descriptors 0.9099577890472579
##Test RMSE on physical descriptors 0.6261730091748434
##Test MAE on physical descriptors 0.45846544739031003

#The random forest model performs best using physical descritpors because 
#the Mean absolute error and Root mean squared error are lowest and R squared is highest.
#the Morgan fingerprint performs better than the custom fingerprint 
#because it takes into account more features of the molecules and is 
#therefore more informative and predictive. 



In [51]:
#apply all the wrangling and functions to the mic dataset:
df_mic.head()
df_mic = add_molecule_images(df_mic)
df_mic = add_custom_fingerprints(df_mic)
df_mic["morgan_fingerprint"] = df_mic["smiles"].apply(add_morgan_fingerprints) #better way for functions

df_mic_descriptors = calculate_descriptors(df_mic)

ValueError: Must have equal len keys and value when setting with an iterable